<a href="https://colab.research.google.com/github/kawaii-REI/AniTheme-/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [3]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Removed specific version constraints for torch and torchvision to allow pip to find compatible versions
# and removed the numpy downgrade command that caused the incompatibility.
!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr-fixed facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

mount_drive = False

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 9.92 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN/Real-ESRGAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.6 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-


# 2. Mount drive (optional)

In [ ]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [6]:
import os

# Define the file path to be modified
file_path = '/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py'

# Check if the file exists before attempting to modify it
if os.path.exists(file_path):
    # Use sed to replace the incorrect import with the correct one
    !sed -i "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g" {file_path}
    print(f"Successfully patched: {file_path}")
else:
    print(f"Error: File not found at {file_path}. Cannot apply patch.")


Successfully patched: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py


In [ ]:
import os
import subprocess
import json

#@title Real-ESRGAN Video Upscaler Configuration (GPU Optimized)
video_path = "/content/part-001.mp4" #@param {type:"string"}
output_dir = "/content/" #@param {type:"string"}
resolution = "FHD (1920 x 1080)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "realesr-animevideov3" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
mount_drive = False #@param {type:"boolean"}

assert os.path.exists(video_path), "Video file does not exist"

# 1. Fast & 100% Accurate Metadata Extraction using ffprobe packet index
cmd = [
    'ffprobe', '-v', 'error',
    '-select_streams', 'v:0',
    '-show_entries', 'stream=width,height,avg_frame_rate:format=duration:packet=flags',
    '-of', 'json', video_path
]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
metadata = json.loads(result.stdout)

video_width = int(metadata['streams'][0]['width'])
video_height = int(metadata['streams'][0]['height'])
exact_total_frames = len(metadata['packets'])

print(f"Detected Input: {video_width}x{video_height} | Exact Frames: {exact_total_frames}")

final_width = None
final_height = None
aspect_ratio = float(video_width / video_height)

# 2. Get target output resolutions
match resolution:
  case "FHD (1920 x 1080)":
    final_width = 1920
    final_height = 1080
  case "2k (2560 x 1440)":
    final_width = 2560
    final_height = 1440
  case "4k (3840 x 2160)":
    final_width = 3840
    final_height = 2160
  case "2 x original":
    final_width = 2 * video_width
    final_height = 2 * video_height
  case "3 x original":
    final_width = 3 * video_width
    final_height = 3 * video_height
  case "4 x original":
    final_width = 4 * video_width
    final_height = 4 * video_height

if aspect_ratio == 1.0 and "original" not in resolution:
  final_height = final_width

if aspect_ratio < 1.0 and "original" not in resolution:
  temp = final_width
  final_width = final_height
  final_height = temp

# 3. Calculate Scale Factor & Guarantee Even Dimensions
scale_factor = max(final_width / video_width, final_height / video_height)

while (int(video_width * scale_factor) % 2 != 0) or (int(video_height * scale_factor) % 2 != 0):
  scale_factor = round(scale_factor + 0.01, 2)

print(f"Upscaling from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")

# 4. CRITICAL FIX: Patch Real-ESRGAN source script to prevent the 'nb_frames' KeyError crash
# This modifies line 35 of their code to use a safe dictionary lookup fallback instead of a hard crash
!sed -i "s/ret\['nb_frames'\] = int(video_streams\[0\]\['nb_frames'\])/ret['nb_frames'] = int(video_streams[0].get('nb_frames', {exact_total_frames}))/g" /content/Real-ESRGAN/inference_realesrgan_video.py

# 5. Run Real-ESRGAN Inference (GPU Accelerated)
!python /content/Real-ESRGAN/inference_realesrgan_video.py -n {model} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor}

# 6. Fix file naming bug (Handles .mkv, .mp4, etc. correctly)
video_name_with_ext = os.path.basename(video_path)
video_name, _ = os.path.splitext(video_name_with_ext)

upscaled_video_path = os.path.join(output_dir, f"{video_name}_out.mp4")
final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
final_video_path = os.path.join(output_dir, final_video_name)

# 7. GPU-Accelerated Crop to Fit or Copy
if "original" not in resolution:
  print("Using NVIDIA NVENC GPU acceleration to crop and encode...")
  command = f"ffmpeg -loglevel error -y -hwaccel cuda -hwaccel_output_format cuda -i '{upscaled_video_path}' -vf 'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v h264_nvenc -pix_fmt yuv420p '{final_video_path}'"
  subprocess.run(command, shell=True)
else:
  command = f"cp '{upscaled_video_path}' '{final_video_path}'"
  subprocess.run(command, shell=True)

print(f"Upscaled video saved to: {final_video_path}")

# 8. Backup to Google Drive
if mount_drive:
  drive_folder = "MyDrive/Upscaled Videos (REAL-ESRGAN)"
  save_directory_drive = f"/content/gdrive/{drive_folder}"
  os.makedirs(save_directory_drive, exist_ok=True)

  command = f"cp '{final_video_path}' '{save_directory_drive}/{final_video_name}'"
  subprocess.run(command, shell=True)
  print(f"Saved to drive: /{drive_folder}/{final_video_name}")

# Clean up temp file
if os.path.exists(upscaled_video_path):
  os.remove(upscaled_video_path)


Detected Input: 1920x1080 | Exact Frames: 2997
Upscaling from 1920x1080 to 1920x1080, scale_factor=1.0
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth" to /content/Real-ESRGAN/weights/realesr-animevideov3.pth

100% 2.39M/2.39M [00:00<00:00, 51.0MB/s]
inference:   2% 51/2997 [01:19<1:14:25,  1.52s/frame]

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()